In [1]:
from pathlib import Path
import pandas as pd
from collections import defaultdict
import numpy as np
from rdkit import Chem
import pickle
from quartz.data import MoleculeData
from quartz.grids import create_grid, to_dxgrid

In [ ]:
# Linux paths
# Load PDBs from directory and map to Excel Data
ligands_dir = Path("~/Documents/Major/Ligands/all_ligs").expanduser()
prop_dir = Path("sample_data/FXa_PDB_RESP_charges")
excel_path = Path("~/Documents/Major/Ligands/full_database_ligands.xlsx").expanduser()

In [ ]:
# Load PDBs from directory and map to Excel Data
ligands_dir = Path("~/Documents/Uni/Major/Ligands/all_ligs").expanduser()
prop_dir = Path("~/Documents/Uni/Major/CSV_props").expanduser()
excel_path = Path(
    "~/Documents/Uni/Major/Ligands/full_database_ligands.xlsx"
).expanduser()

In [ ]:
# Read Excel file
print(f"Reading Excel file: {excel_path}")
fx_data = pd.read_excel(excel_path)
# Clean column names by stripping whitespace
fx_data.columns = fx_data.columns.str.strip()

pdb_files = list(ligands_dir.glob("*_out_reduced.pdb"))

mol_data_list = []

print(f"Found {len(pdb_files)} PDB files in {ligands_dir}")

for pdb_file in pdb_files:
    # Extract PDB ID from filename (e.g. 1EZQ_out_reduced.pdb -> 1EZQ)
    pdb_id = pdb_file.name.split("_")[0]
    print(f"Processing PDB ID: {pdb_id}")

    # Path to charge file
    try:
        prop_file = prop_dir.glob(f"{pdb_id}*.csv")
    # print(f"Looking for charge file: {charge_file}")
    except IndexError:
        print(f"Prop file for {pdb_id} not found.")

    extra = defaultdict(float)

    with open(pdb_file, "r") as f:
        pdb_content = f.readlines()
        for line in pdb_content:
            # print(line)
            if line.startswith("REMARK"):
                # print(line.strip())
                if "minimizedAffinity" in line:
                    extra["minimizedAffinity"] = float(line.split()[-1])
                    # print("Minimized Affinity:", extra['minimizedAffinity'])
                if "CNNscore" in line:
                    extra["CNNscore"] = float(line.split()[-1])
                    # print("CNN Score:", extra['CNNscore'])
                if "CNNaffinity" in line:
                    extra["CNNaffinity"] = float(line.split()[-1])
                    # print("CNN Affinity:", extra['CNNaffinity'])

            if (
                extra["minimizedAffinity"]
                and extra["CNNscore"]
                and extra["CNNaffinity"]
            ):
                break

    # Get pKi and Ki for the PDB ID
    # Assuming column is 'PDB ID' after stripping
    row = fx_data[fx_data["PDB ID"].astype(str).str.strip() == pdb_id]

    if row.empty:
        print(f"No FX Data found for {pdb_id}")
        continue

    try:
        ki = float(row["Ki (nM)"].values[0])
    except ValueError:
        print(f"Invalid Ki value for {pdb_id}, skipping...")
        continue
    # print("Ki:", ki)
    if "pKi" in row.columns:
        pki = float(row["pKi"].values[0])
    else:
        pki = -np.log10(ki * 1e-9)  # Convert Ki in nM to pKi
    # print("pKi:", pki)

    # Store ki and pki in extra dict
    extra["ki"] = ki
    extra["pki"] = pki

    # Load Mol from PDB
    mol = Chem.MolFromPDBFile(str(pdb_file), removeHs=False)
    if mol is None:
        print(f"Failed to load PDB: {pdb_file.name}")
        continue

    if not prop_file.exists():
        print(f"Charge file not found for {pdb_id}")
        continue

    charges_df = pd.read_csv(prop_file)

    # Check atom count and charge assignment
    if len(charges_df) != mol.GetNumAtoms():
        print(
            f"Atom count mismatch for {pdb_id}: mol {mol.GetNumAtoms()} vs csv {len(charges_df)}"
        )
        continue

    # Assign RESP charges
    charge_sum = 0.0
    for idx, row_charge in charges_df.iterrows():
        atom = mol.GetAtomWithIdx(idx)
        resp_charge = float(row_charge["RESP"])
        atom.SetProp("RESP", str(resp_charge))
        charge_sum += resp_charge

        # Verify assignment
        if abs(float(atom.GetProp("RESP")) - resp_charge) > 1e-6:
            print(f"Charge assignment failed for {pdb_id} atom {idx}")

    # print(f"Processed {pdb_id}: Total Charge = {charge_sum:.4f}")

    # Create MoleculeData (ki and pki are now in extra dict)
    md = MoleculeData(
        atomic_numbers=np.array([a.GetAtomicNum() for a in mol.GetAtoms()]),
        coordinates=mol.GetConformer().GetPositions(),
        res={},
        smiles=Chem.MolToSmiles(mol),
        rdmol=mol,
        pdb_code=pdb_id,
        filename=pdb_id,
        extra=extra,
    )
    mol_data_list.append(md)

print(f"Prepared {len(mol_data_list)} MoleculeData objects")

Reading Excel file: C:\Users\friedrich\Documents\Uni\Major\Ligands\full_database_ligands.xlsx
Found 1498 PDB files in C:\Users\friedrich\Documents\Uni\Major\Ligands\all_ligs
Processing PDB ID: 0001
Processing PDB ID: 0002
Processing PDB ID: 0003
Processing PDB ID: 0004
Processing PDB ID: 0005
Processing PDB ID: 0006
Processing PDB ID: 0007
Processing PDB ID: 0008
Processing PDB ID: 0009
Processing PDB ID: 0010
Processing PDB ID: 0011
Processing PDB ID: 0012
Processing PDB ID: 0013
Processing PDB ID: 0014
Processing PDB ID: 0015
Processing PDB ID: 0016
Processing PDB ID: 0017
Processing PDB ID: 0018
Processing PDB ID: 0019
Processing PDB ID: 0020
Processing PDB ID: 0021
Processing PDB ID: 0022
Processing PDB ID: 0023
Charge file not found for 0023, skipping...
Processing PDB ID: 0024
Processing PDB ID: 0025
Processing PDB ID: 0026
Processing PDB ID: 0027
Processing PDB ID: 0028
Processing PDB ID: 0029
Processing PDB ID: 0030
Processing PDB ID: 0031
Processing PDB ID: 0032
Processing PDB

[11:15:48] Explicit valence for atom # 14 C, 5, is greater than permitted


Processing PDB ID: 0330
Processing PDB ID: 0331
Processing PDB ID: 0332
Processing PDB ID: 0333
Processing PDB ID: 0334
Processing PDB ID: 0335
Processing PDB ID: 0336
Processing PDB ID: 0337
Processing PDB ID: 0338
Processing PDB ID: 0339
Processing PDB ID: 0340
Processing PDB ID: 0341
Processing PDB ID: 0342
Processing PDB ID: 0343
Processing PDB ID: 0344
Processing PDB ID: 0345
Processing PDB ID: 0346
Processing PDB ID: 0347
Processing PDB ID: 0348
Processing PDB ID: 0349
Processing PDB ID: 0350
Processing PDB ID: 0351
Processing PDB ID: 0352
Processing PDB ID: 0353
Processing PDB ID: 0354
Processing PDB ID: 0355
Processing PDB ID: 0356
Processing PDB ID: 0357
Processing PDB ID: 0358
Processing PDB ID: 0359
Processing PDB ID: 0360
Processing PDB ID: 0361
Processing PDB ID: 0362
Charge file not found for 0362, skipping...
Processing PDB ID: 0363
Processing PDB ID: 0364
Processing PDB ID: 0365
Processing PDB ID: 0366
Processing PDB ID: 0367
Processing PDB ID: 0368
Processing PDB ID: 0

[11:15:52] Explicit valence for atom # 7 C, 5, is greater than permitted


Processing PDB ID: 0460
Processing PDB ID: 0461
Processing PDB ID: 0462
Processing PDB ID: 0463
Processing PDB ID: 0464
Processing PDB ID: 0465
Processing PDB ID: 0466
Processing PDB ID: 0467
Invalid Ki value for 0467, skipping...
Processing PDB ID: 0468
Processing PDB ID: 0469
Invalid Ki value for 0469, skipping...
Processing PDB ID: 0470
Processing PDB ID: 0471
Processing PDB ID: 0472
Invalid Ki value for 0472, skipping...
Processing PDB ID: 0473
Processing PDB ID: 0474
Processing PDB ID: 0475
Processing PDB ID: 0476
Processing PDB ID: 0477
Processing PDB ID: 0478
Processing PDB ID: 0479
Processing PDB ID: 0480
Processing PDB ID: 0481
Invalid Ki value for 0481, skipping...
Processing PDB ID: 0482
Processing PDB ID: 0483
Processing PDB ID: 0484
Processing PDB ID: 0485
Processing PDB ID: 0486
Processing PDB ID: 0487
Processing PDB ID: 0488
Processing PDB ID: 0489
Processing PDB ID: 0490
Processing PDB ID: 0491
Processing PDB ID: 0492
Processing PDB ID: 0493
Processing PDB ID: 0494
Proc

[11:15:57] Explicit valence for atom # 23 O, 3, is greater than permitted


... (5 outputs left out)

[11:16:04] Explicit valence for atom # 22 C, 5, is greater than permitted


Processing PDB ID: 0860
Processing PDB ID: 0861
Processing PDB ID: 0862
Processing PDB ID: 0863
Processing PDB ID: 0864
Processing PDB ID: 0865
Processing PDB ID: 0866
Processing PDB ID: 0867
Processing PDB ID: 0868
Processing PDB ID: 0869
Processing PDB ID: 0870
Processing PDB ID: 0871
Processing PDB ID: 0872
Charge file not found for 0872, skipping...
Processing PDB ID: 0873
Processing PDB ID: 0874
Processing PDB ID: 0875
Processing PDB ID: 0876
Processing PDB ID: 0877
Processing PDB ID: 0878
Processing PDB ID: 0879
Processing PDB ID: 0880
Processing PDB ID: 0881
Processing PDB ID: 0882
Processing PDB ID: 0883
Processing PDB ID: 0884
Processing PDB ID: 0885
Processing PDB ID: 0886
Processing PDB ID: 0887
Processing PDB ID: 0888
Processing PDB ID: 0889
Processing PDB ID: 0890
Charge file not found for 0890, skipping...
Processing PDB ID: 0891
Processing PDB ID: 0892
Processing PDB ID: 0893
Processing PDB ID: 0894
Processing PDB ID: 0895
Processing PDB ID: 0896
Processing PDB ID: 0897


[11:16:16] Explicit valence for atom # 21 C, 5, is greater than permitted


Processing PDB ID: 4BTT
Processing PDB ID: 4BTU
Processing PDB ID: 4Y6D
Processing PDB ID: 4Y71
Processing PDB ID: 4Y76
Processing PDB ID: 4Y79
Processing PDB ID: 4Y7A
Processing PDB ID: 4Y7B
Processing PDB ID: 4ZHA
Prepared 1263 MoleculeData objects


In [6]:
# Generate grids
print("Generating grids...")
grids = create_grid(mol_data_list, props=["RESP", "polarizability", "HOMO_contrib", "LUMO_contrib", "effectiveV"], size=0.5, threshold={"RESP": 0.001, "polarizability": 0.01, "HOMO_contrib": 0.00001, "LUMO_contrib": 0.00001, "effectiveV": 0.1})

Generating grids...
Grid lengths:  28.0 32.0 30.0
Origin:  -7.555 26.239 44.007
Cell count:  56 64 60
Processing:  0001


KeyError: 'polarizability'

In [6]:
# Save to pkl
output_file = "grids.pkl"
with open(output_file, "wb") as f:
    pickle.dump(grids, f)

In [ ]:
# Save to DX
print("Saving to DX format...")
to_dxgrid(grids)